# Домашнее задание № 8

## Задание 1 (4 балла) 

Обучите 2 модели похожую по архитектуре на модель из ULMFit для задачи классификации текста (датасет - lenta_40k )
В моделях должно быть как минимум два рекуррентных слоя, а финальный вектор для классификации составляться из последнего состояния RNN (так делалось в семинаре), а также AveragePooling и MaxPooling из всех векторов последовательности (конкатенируйте последнее состояния и результаты пулинга). В первой модели используйте обычные слои, а во второй Bidirectional. Рассчитайте по классовую точность/полноту/f-меру для каждой из модели (результаты не должны быть совсем близкие к нулю после обучения на хотя бы нескольких эпохах). 

In [6]:
from datasets import load_dataset
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
from torch.utils.data import DataLoader, TensorDataset
from collections import Counter
from sklearn.metrics import classification_report
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

In [3]:
path = r"C:\Users\schig\Downloads\lenta_40k.csv"
df = pd.read_csv(path)

df.head()

,text,topic
0,Россия должна сотрудничать с Всемирным антидоп...,Спорт
1,Уголовный суд Кувейта 28 июня освободил под за...,Мир
2,Французский журнал Charlie Hebdo опубликовал н...,Интернет и СМИ
3,В Петербурге в доме № 53 по улице Лени Голиков...,Россия
4,"В московском аэропорту ""Домодедово"" задержан г...",Россия


In [8]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

label_encoder = LabelEncoder()
df["label_id"] = label_encoder.fit_transform(df["topic"])

label_encoder = LabelEncoder()
df["label_id"] = label_encoder.fit_transform(df["topic"])
label2id = dict(zip(label_encoder.classes_, range(len(label_encoder.classes_))))
id2label = dict(enumerate(label_encoder.classes_))

train_df, val_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df["label_id"])

vocab = Counter()
for text in train_df["text"]:
    vocab.update(text.lower().split())

word2id = {"PAD": 0, "UNK": 1}
for w in vocab:
    word2id[w] = len(word2id)

def encode(texts, word2id):
    return [[word2id.get(w, 1) for w in t.lower().split()] for t in texts]

X_train = encode(train_df["text"], word2id)
X_val = encode(val_df["text"], word2id)
y_train = train_df["label_id"].values
y_val = val_df["label_id"].values

MAX_LEN = 100
def pad(seqs, max_len):
    return np.array([s[:max_len] + [0] * max(0, max_len - len(s)) for s in seqs])

X_train = pad(X_train, MAX_LEN)
X_val = pad(X_val, MAX_LEN)

train_loader = DataLoader(TensorDataset(torch.LongTensor(X_train), torch.LongTensor(y_train)), batch_size=64, shuffle=True)
val_loader = DataLoader(TensorDataset(torch.LongTensor(X_val), torch.LongTensor(y_val)), batch_size=64)

class TextClassifierStacked(nn.Module):
    def __init__(self, vocab_size, emb_dim=30, hidden_size=128, output_dim=10, num_layers=2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
        self.rnn = nn.LSTM(emb_dim, hidden_size, num_layers=num_layers, batch_first=True, dropout=0.2)
        self.fc = nn.Linear(hidden_size * 3, output_dim)
    def forward(self, text):
        embedded = self.embedding(text)
        output, (h, c) = self.rnn(embedded)
        last_hidden = h[-1]
        avg_pool = output.mean(dim=1)
        max_pool, _ = output.max(dim=1)
        features = torch.cat([last_hidden, avg_pool, max_pool], dim=1)
        return self.fc(features)

class TextClassifierBi(nn.Module):
    def __init__(self, vocab_size, emb_dim=30, hidden_size=128, output_dim=10, num_layers=2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
        self.rnn = nn.LSTM(emb_dim, hidden_size, num_layers=num_layers, batch_first=True, bidirectional=True, dropout=0.2)
        self.fc = nn.Linear(hidden_size * 2 * 3, output_dim)
    def forward(self, text):
        embedded = self.embedding(text)
        output, (h, c) = self.rnn(embedded)
        h_forward = h[-2]
        h_backward = h[-1]
        last_hidden = torch.cat([h_forward, h_backward], dim=1)
        avg_pool = output.mean(dim=1)
        max_pool, _ = output.max(dim=1)
        features = torch.cat([last_hidden, avg_pool, max_pool], dim=1)
        return self.fc(features)

def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0
    all_preds, all_labels = [], []
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        all_preds.extend(logits.argmax(dim=1).cpu().numpy())
        all_labels.extend(y.cpu().numpy())
    f1 = classification_report(all_labels, all_preds, output_dict=True, zero_division=0)["macro avg"]["f1-score"]
    return total_loss / len(loader), f1

def evaluate(model, loader, criterion):
    model.eval()
    total_loss = 0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            loss = criterion(logits, y)
            total_loss += loss.item()
            all_preds.extend(logits.argmax(dim=1).cpu().numpy())
            all_labels.extend(y.cpu().numpy())
    f1 = classification_report(all_labels, all_preds, output_dict=True, zero_division=0)["macro avg"]["f1-score"]
    return total_loss / len(loader), f1

model = TextClassifierStacked(len(word2id), emb_dim=30, hidden_size=128, output_dim=len(label2id)).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

for epoch in range(5):
    train_loss, train_f1 = train_epoch(model, train_loader, optimizer, criterion)
    val_loss, val_f1 = evaluate(model, val_loader, criterion)
    print(f"Stacked LSTM Epoch {epoch+1}: Train F1={train_f1:.4f} | Val F1={val_f1:.4f}")

model = TextClassifierBi(len(word2id), emb_dim=30, hidden_size=128, output_dim=len(label2id)).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

for epoch in range(5):
    train_loss, train_f1 = train_epoch(model, train_loader, optimizer, criterion)
    val_loss, val_f1 = evaluate(model, val_loader, criterion)
    print(f"BiLSTM Epoch {epoch+1}: Train F1={train_f1:.4f} | Val F1={val_f1:.4f}")

def per_class_metrics(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            p = model(x).argmax(dim=1)
            all_preds.extend(p.cpu().numpy())
            all_labels.extend(y.cpu().numpy())
    labels = list(range(len(label_encoder.classes_)))  
    print(classification_report(
        all_labels,
        all_preds,
        labels=labels,
        target_names=label_encoder.classes_,
        zero_division=0
    ))

Using device: cpu
Stacked LSTM Epoch 1: Train F1=0.0741 | Val F1=0.1432
Stacked LSTM Epoch 2: Train F1=0.1782 | Val F1=0.2169
Stacked LSTM Epoch 3: Train F1=0.2820 | Val F1=0.2876
Stacked LSTM Epoch 4: Train F1=0.3863 | Val F1=0.3567
Stacked LSTM Epoch 5: Train F1=0.4731 | Val F1=0.3636
BiLSTM Epoch 1: Train F1=0.1344 | Val F1=0.2134
BiLSTM Epoch 2: Train F1=0.2930 | Val F1=0.2933
BiLSTM Epoch 3: Train F1=0.4008 | Val F1=0.3389
BiLSTM Epoch 4: Train F1=0.5038 | Val F1=0.3654
BiLSTM Epoch 5: Train F1=0.6246 | Val F1=0.3616


## Задание 2 (6 баллов)


На данных википедии (wikiann) обучите и сравните 3 модели:

модель в которой как минимум два рекуррентных слоя, причем один из них GRU, а другой LSTM
модель в которой как минимум 3 рекуррентных слоя идут друг за другом и при этом 2-ой и 3-й слои еще имеют residual connection к изначальным эмбедингам. Для того, чтобы сделать residual connection вам нужно будет использовать одинаковую размерность эмбедингов и количество unit'ов в RNN слоях, чтобы их можно было просуммировать
модель в которой будут и рекуррентные и сверточные слои (как минимум 2 rnn и как минимум 2 cnn слоя). В cnn слоях будьте аккуратны с укорачиванием последовательности и используйте паддинг
Сравните качество по метрикам (точность/полнота/f-мера). Также придумайте несколько сложных примеров и проверьте, какие сущности определяет каждая из моделей.

In [14]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from tqdm.notebook import tqdm
from datasets import load_dataset
from collections import Counter
from sklearn.metrics import precision_recall_fscore_support

In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

dataset = load_dataset("unimelb-nlp/wikiann", "ru")

In [8]:
PAD_TOKEN = "<PAD>"
UNK_TOKEN = "<UNK>"

token_counter = Counter()
for split in ["train", "validation", "test"]:
    for tokens in dataset[split]["tokens"]:
        token_counter.update(tokens)

word2id = {PAD_TOKEN: 0, UNK_TOKEN: 1}
for token in token_counter:
    word2id[token] = len(word2id)

id2word = {i: w for w, i in word2id.items()}

label_list = dataset["train"].features["ner_tags"].feature.names
label2id = {l: i for i, l in enumerate(label_list)}
id2label = {i: l for l, i in label2id.items()}

In [10]:
MAX_LEN = 200
BATCH_SIZE = 256
EMB_DIM = 128
HIDDEN_SIZE = 128
EPOCHS = 5

def encode_tokens(tokens):
    return [word2id.get(t, word2id[UNK_TOKEN]) for t in tokens]

def pad_sequence(seq, max_len, pad_value):
    if len(seq) > max_len:
        return seq[:max_len]
    return seq + [pad_value] * (max_len - len(seq))

def prepare_split(split):
    X, y = [], []
    for tokens, tags in zip(dataset[split]["tokens"], dataset[split]["ner_tags"]):
        X.append(pad_sequence(encode_tokens(tokens), MAX_LEN, 0))
        y.append(pad_sequence(tags, MAX_LEN, -100))
    return torch.LongTensor(X), torch.LongTensor(y)

X_train, y_train = prepare_split("train")
X_valid, y_valid = prepare_split("validation")

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True)
valid_loader = DataLoader(TensorDataset(X_valid, y_valid), batch_size=BATCH_SIZE, shuffle=False)

In [11]:
class GRU_LSTM_Model(nn.Module):
    def __init__(self, vocab_size, emb_dim, hidden_size, num_labels):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
        self.gru = nn.GRU(emb_dim, hidden_size, batch_first=True)
        self.lstm = nn.LSTM(hidden_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, num_labels)
    def forward(self, x):
        x = self.embedding(x)
        x, _ = self.gru(x)
        x, _ = self.lstm(x)
        return self.fc(x)

class Residual_LSTM_Model(nn.Module):
    def __init__(self, vocab_size, emb_dim, hidden_size, num_labels):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
        self.proj = nn.Linear(emb_dim, hidden_size)
        self.rnn1 = nn.LSTM(hidden_size, hidden_size, batch_first=True)
        self.rnn2 = nn.LSTM(hidden_size, hidden_size, batch_first=True)
        self.rnn3 = nn.LSTM(hidden_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, num_labels)
    def forward(self, x):
        x = self.embedding(x)
        x = self.proj(x)
        out1, _ = self.rnn1(x)
        out2, _ = self.rnn2(out1)
        out2 = out2 + x
        out3, _ = self.rnn3(out2)
        out3 = out3 + x
        return self.fc(out3)

In [12]:
class RNN_CNN_Model(nn.Module):
    def __init__(self, vocab_size, emb_dim, hidden_size, num_labels):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
        self.lstm = nn.LSTM(emb_dim, hidden_size, batch_first=True, bidirectional=True)
        self.gru = nn.GRU(hidden_size * 2, hidden_size, batch_first=True, bidirectional=True)
        self.conv1 = nn.Conv1d(hidden_size * 2, hidden_size, kernel_size=3, padding=1)
        self.conv2 = nn.Conv1d(hidden_size, hidden_size, kernel_size=3, padding=1)
        self.fc = nn.Linear(hidden_size, num_labels)
    def forward(self, x):
        x = self.embedding(x)
        x, _ = self.lstm(x)
        x, _ = self.gru(x)
        x = x.transpose(1, 2)
        x = torch.relu(self.conv1(x))
        x = torch.relu(self.conv2(x))
        x = x.transpose(1, 2)
        return self.fc(x)

def train_epoch(model, loader, optimizer, criterion):
    model.train()
    losses = []
    for x, y in tqdm(loader, leave=False):
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits.view(-1, logits.size(-1)), y.view(-1))
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    return np.mean(losses)

def evaluate(model, loader):
    model.eval()
    all_preds, all_true = [], []
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            logits = model(x)
            preds = logits.argmax(-1).cpu().numpy()
            y = y.numpy()
            mask = y != -100
            all_preds.extend(preds[mask])
            all_true.extend(y[mask])
    p, r, f, _ = precision_recall_fscore_support(all_true, all_preds, average="micro")
    return p, r, f

In [15]:
models = {
    "GRU+LSTM": GRU_LSTM_Model(len(word2id), EMB_DIM, HIDDEN_SIZE, len(label2id)),
    "Residual_LSTM": Residual_LSTM_Model(len(word2id), EMB_DIM, HIDDEN_SIZE, len(label2id)),
    "RNN+CNN": RNN_CNN_Model(len(word2id), EMB_DIM, HIDDEN_SIZE, len(label2id)),
}

for name, model in models.items():
    model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss(ignore_index=-100)
    for epoch in range(EPOCHS):
        train_epoch(model, train_loader, optimizer, criterion)
    p, r, f = evaluate(model, valid_loader)
    print(name, "Precision:", round(p, 4), "Recall:", round(r, 4), "F1:", round(f, 4))

  0%|          | 0/79 [00:00<?, ?it/s]

  0%|          | 0/79 [00:00<?, ?it/s]

  0%|          | 0/79 [00:00<?, ?it/s]

  0%|          | 0/79 [00:00<?, ?it/s]

  0%|          | 0/79 [00:00<?, ?it/s]

...

  0%|          | 0/79 [00:00<?, ?it/s]

  0%|          | 0/79 [00:00<?, ?it/s]

  0%|          | 0/79 [00:00<?, ?it/s]

  0%|          | 0/79 [00:00<?, ?it/s]

RNN+CNN Precision: 0.8638 Recall: 0.8638 F1: 0.8638


In [16]:
examples = [
    ["Трамп", "призвал", "Европу", "сконцентрироваться", "на", "Украине", "а", "не", "на", "Гренландии", "."],
    ["Лурье", "получила", "ключи", "от", "бывшей", "квартиры", "Долиной", "."],
    ["Путин", "традиционно", "окунулся", "в", "прорубь", "на", "Крещение", "Господне", "."],
]

def predict(model, tokens):
    ids = pad_sequence(encode_tokens(tokens), MAX_LEN, 0)
    x = torch.LongTensor([ids]).to(device)
    logits = model(x)
    preds = logits.argmax(-1).cpu().numpy()[0][:len(tokens)]
    return list(zip(tokens, [id2label[p] for p in preds]))

for name, model in models.items():
    print("\n", name)
    for ex in examples:
        print(predict(model, ex))


 GRU+LSTM
[('Трамп', 'O'), ('призвал', 'O'), ('Европу', 'O'), ('сконцентрироваться', 'O'), ('на', 'O'), ('Украине', 'O'), ('а', 'O'), ('не', 'O'), ('на', 'O'), ('Гренландии', 'O'), ('.', 'O')]
[('Лурье', 'O'), ('получила', 'O'), ('ключи', 'O'), ('от', 'O'), ('бывшей', 'O'), ('квартиры', 'O'), ('Долиной', 'O'), ('.', 'O')]
[('Путин', 'B-PER'), ('традиционно', 'I-ORG'), ('окунулся', 'O'), ('в', 'O'), ('прорубь', 'O'), ('на', 'O'), ('Крещение', 'O'), ('Господне', 'O'), ('.', 'O')]

 Residual_LSTM
[('Трамп', 'B-PER'), ('призвал', 'I-ORG'), ('Европу', 'I-ORG'), ('сконцентрироваться', 'I-ORG'), ('на', 'I-ORG'), ('Украине', 'I-ORG'), ('а', 'I-ORG'), ('не', 'I-ORG'), ('на', 'I-ORG'), ('Гренландии', 'I-ORG'), ('.', 'O')]
[('Лурье', 'B-PER'), ('получила', 'I-ORG'), ('ключи', 'I-ORG'), ('от', 'O'), ('бывшей', 'O'), ('квартиры', 'I-ORG'), ('Долиной', 'O'), ('.', 'O')]
[('Путин', 'B-PER'), ('традиционно', 'I-ORG'), ('окунулся', 'I-ORG'), ('в', 'O'), ('прорубь', 'O'), ('на', 'O'), ('Крещение', 'O')

Модель GRU + LSTM лучщая по F1, но качественно работает плохо. В новостных заголовках не распознала персон, фамилии. Практически, везде по нулям, видимо, очень консервативная и если не знает точно, предпочитает о.

Модель Residual LSTM оказалась более чувствительная к именам/фамилиям, но часто превращает всё в ORG, есть сильные BIO-ошибки. Очень пдлохо держит границы.

И модель RNN+CNN находит сущности, хорошо реагирует на локальные паттерны, но часто неверный тип, много ложных сущностей.